In [1]:
# ==========================================
# CELL 1: IMPORTS & SETUP
# ==========================================
import pandas as pd
import numpy as np
import re
import unicodedata
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score
from sklearn.svm import LinearSVC
from lightgbm import LGBMClassifier
from scipy.sparse import hstack, csr_matrix

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
import lightgbm as lgb

# Setup Progress Bar (tqdm)
from tqdm.auto import tqdm
tqdm.pandas() # Kích hoạt progress_apply cho pandas

# Tắt log Optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================
# CELL 2: LOAD DATA
# ==========================================
DATA_DIR = r"D:\nlp\project\CS221\Data\Clean"
TRAIN_PATH = f"{DATA_DIR}/train_clean.csv"
VAL_PATH = f"{DATA_DIR}/val_clean.csv"
TEST_PATH = f"{DATA_DIR}/test_clean.csv"

print("Loading data...")
df_train = pd.read_csv(TRAIN_PATH)
df_val = pd.read_csv(VAL_PATH)
df_test = pd.read_csv(TEST_PATH)

df_train_full = pd.concat([df_train, df_val], ignore_index=True)

print(f"Train size (merged): {len(df_train_full)}")
print(f"Test size: {len(df_test)}")

# Label Encoding
le = LabelEncoder()
y_train_full = le.fit_transform(df_train_full['label'])
y_test = le.transform(df_test['label'])
class_names = le.classes_

# ==========================================
# CELL 3: TEXT PREPROCESSING (WITH PROGRESS)
# ==========================================
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
    text = text.replace('<url>', ' special_token_url ')
    text = text.replace('<username>', ' special_token_user ')
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'[^\w\s:]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Preprocessing data (with progress bar)...")
X_train_full_text = df_train_full['text'].progress_apply(preprocess_text)
X_test_text = df_test['text'].progress_apply(preprocess_text)
print("Preprocessing done.")

# ==========================================
# CELL 4: ADVANCED FEATURE ENGINEERING
# ==========================================
print("Extracting features...")

# 1. Word-level TF-IDF
tfidf_word_vectorizer = TfidfVectorizer(
    ngram_range=(1, 3),       # Bigram & Trigram
    max_features=10000, 
    sublinear_tf=True,
    min_df=2
)

# 2. Char-level TF-IDF
tfidf_char_vectorizer = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3, 5),       # Catches typos & suffixes
    max_features=1000,
    sublinear_tf=True
)

def get_features_adv(text_series, vec_word, vec_char, fit=False):
    if fit:
        tfidf_word = vec_word.fit_transform(text_series)
        tfidf_char = vec_char.fit_transform(text_series)
    else:
        tfidf_word = vec_word.transform(text_series)
        tfidf_char = vec_char.transform(text_series)
    
    word_counts = text_series.apply(lambda x: len(x.split())).values
    word_counts_sparse = csr_matrix(word_counts).T
    
    avg_word_len = text_series.apply(lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0).values
    avg_word_len_sparse = csr_matrix(avg_word_len).T
    
    return hstack([tfidf_word, tfidf_char, word_counts_sparse, avg_word_len_sparse])

X_train_features = get_features_adv(X_train_full_text, tfidf_word_vectorizer, tfidf_char_vectorizer, fit=True)
X_test_features = get_features_adv(X_test_text, tfidf_word_vectorizer, tfidf_char_vectorizer, fit=False)

if hasattr(X_train_features, "tocsr"):
    X_train_features = X_train_features.tocsr()
if hasattr(X_test_features, "tocsr"):
    X_test_features = X_test_features.tocsr()

print(f"Feature shape: {X_train_features.shape}")

# ==========================================
# CELL 5: OPTUNA TUNING FOR SVM ONLY (FAST)
# ==========================================
def objective_svm(trial, X, y):
    params = {
        'C': trial.suggest_float('C', 1e-4, 100, log=True),
        'class_weight': 'balanced',
        'max_iter': 3000
    }
    model = LinearSVC(**params)
    scores = cross_val_score(model, X, y, cv=5, scoring='f1_macro', n_jobs=-1)
    return scores.mean()

print("\n=========================")
print("TUNING LINEAR SVM (Optuna)")
print("=========================")

N_TRIALS_SVM = 15
study_svm = optuna.create_study(direction='maximize')
study_svm.optimize(
    lambda t: objective_svm(t, X_train_features, y_train_full),
    n_trials=N_TRIALS_SVM,
    show_progress_bar=True
)

print(f"\n[SVM DONE] Best F1: {study_svm.best_value:.4f}")
print("Best params:", study_svm.best_params)

# ==========================================
# CELL 6: MANUAL TRAINING FOR LIGHTGBM (NO OPTUNA)
# ==========================================
print("\n=========================")
print("TRAINING LIGHTGBM (Manual + Early Stopping)")
print("=========================")

# 1. Chia nhỏ dữ liệu train để làm validation set cho Early Stopping
X_train_manual, X_val_manual, y_train_manual, y_val_manual = train_test_split(
    X_train_features, y_train_full, test_size=0.1, random_state=42, stratify=y_train_full
)

# 2. Tham số "Best Practice" cho Text Classification với LGBM
lgbm_params = {
    'objective': 'multiclass',
    'metric': 'multi_logloss',
    'num_class': len(class_names),
    'n_estimators': 2000,       # Số cây cao, ES sẽ tự cắt
    'learning_rate': 0.1,       # Learning rate mặc định khá tốt
    'num_leaves': 31,
    'max_depth': -1,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'class_weight': 'balanced',
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

print(f"Training LGBM on {X_train_manual.shape[0]} samples...")
model_lgbm = LGBMClassifier(**lgbm_params)

# 3. Train với Early Stopping
model_lgbm.fit(
    X_train_manual, y_train_manual,
    eval_set=[(X_val_manual, y_val_manual)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=100) # In log mỗi 100 cây để xem tiến độ
    ]
)
print("LGBM Training finished.")

# ==========================================
# CELL 7: FINAL EVALUATION
# ==========================================
print("\n" + "="*40)
print("FINAL EVALUATION ON TEST SET")
print("="*40)

results = []

# --- 1. Evaluate SVM ---
print("\n[1/2] Evaluating Linear SVM...")
best_svm = LinearSVC(**study_svm.best_params, class_weight='balanced')
best_svm.fit(X_train_features, y_train_full)

y_pred_svm = best_svm.predict(X_test_features)
f1_svm = f1_score(y_test, y_pred_svm, average='macro')
print(f"SVM F1-Macro: {f1_svm:.4f}")

results.append({'Model': 'Linear SVM', 'F1-Macro': f1_svm})

# --- 2. Evaluate LightGBM ---
print("\n[2/2] Evaluating LightGBM...")
y_pred_lgbm = model_lgbm.predict(X_test_features)
f1_lgbm = f1_score(y_test, y_pred_lgbm, average='macro')
print(f"LGBM F1-Macro: {f1_lgbm:.4f}")

results.append({'Model': 'LightGBM (Manual)', 'F1-Macro': f1_lgbm})

# ==========================================
# CELL 8: SUMMARY & REPORT
# ==========================================
df_results = pd.DataFrame(results).sort_values(by='F1-Macro', ascending=False)

print("\n=== SUMMARY ===")
display(df_results)

# Chi tiết báo cáo của model tốt nhất
best_model_name = df_results.iloc[0]['Model']
print(f"\n=== CLASSIFICATION REPORT (BEST: {best_model_name}) ===")

if best_model_name == 'Linear SVM':
    print(classification_report(y_test, y_pred_svm, target_names=class_names, digits=4))
else:
    print(classification_report(y_test, y_pred_lgbm, target_names=class_names, digits=4))

Loading data...
Train size (merged): 39544
Test size: 9887
Preprocessing data (with progress bar)...


  0%|          | 0/39544 [00:00<?, ?it/s]

  0%|          | 0/9887 [00:00<?, ?it/s]

Preprocessing done.
Extracting features...
Feature shape: (39544, 11002)

TUNING LINEAR SVM (Optuna)


  0%|          | 0/15 [00:00<?, ?it/s]


[SVM DONE] Best F1: 0.7749
Best params: {'C': 0.11192481872624284}

TRAINING LIGHTGBM (Manual + Early Stopping)
Training LGBM on 35589 samples...
[100]	valid_0's multi_logloss: 0.513216
[200]	valid_0's multi_logloss: 0.494022
LGBM Training finished.

FINAL EVALUATION ON TEST SET

[1/2] Evaluating Linear SVM...
SVM F1-Macro: 0.7758

[2/2] Evaluating LightGBM...
LGBM F1-Macro: 0.7862

=== SUMMARY ===


,Model,F1-Macro
1,LightGBM (Manual),0.786188
0,Linear SVM,0.775766



=== CLASSIFICATION REPORT (BEST: LightGBM (Manual)) ===
              precision    recall  f1-score   support

     Anxiety     0.7980    0.8430    0.8199      1115
  Depression     0.7254    0.6899    0.7072      2902
      Normal     0.9217    0.9311    0.9264      3630
    Suicidal     0.6849    0.6978    0.6913      2240

    accuracy                         0.7975      9887
   macro avg     0.7825    0.7905    0.7862      9887
weighted avg     0.7965    0.7975    0.7968      9887

